In [1]:
import os
import zipfile
import requests
import geopandas as gpd
from io import BytesIO

In [6]:
import os
import mgrs
import geopandas as gpd
from shapely.geometry import Polygon
from pyproj import CRS, Transformer

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

TILES = ["12TYP", "39KUA", "18SUJ", "15TYL", "10SGD", "17PPL"]
OUTPUT_DIR = "mgrs_tiles"
os.makedirs(OUTPUT_DIR, exist_ok=True)

m = mgrs.MGRS()

# ─────────────────────────────────────────────
# FUNCTION: Build polygon for one 100km tile
# ─────────────────────────────────────────────

def get_mgrs_tile_polygon(tile_id):
    """
    Constructs a WGS84 polygon for a 5-character MGRS 100km tile.
    
    Strategy:
      1. Append '0000000000' to get the SW corner at 1m precision
      2. Convert SW corner to lat/lon via mgrs package
      3. Project to the tile's UTM zone
      4. Add 100,000m in easting & northing to get remaining corners
      5. Reproject corners back to WGS84 and build polygon
    """

    # Step 1: SW corner - 10 trailing zeros = 5-digit easting + 5-digit northing
    sw_mgrs_str = tile_id + "0000000000"
    sw_lat, sw_lon = m.toLatLon(sw_mgrs_str)

    # Step 2: Parse UTM zone number and band letter from tile ID
    zone_str = ""
    for ch in tile_id:
        if ch.isdigit():
            zone_str += ch
        else:
            break
    zone_num = int(zone_str)
    band_letter = tile_id[len(zone_str)]  # letter immediately after zone digits

    # Step 3: Determine hemisphere (bands A-M = south, N-Z = north)
    is_south = band_letter.upper() < "N"

    # Step 4: Build UTM CRS for this zone
    utm_crs = CRS.from_dict({"proj": "utm", "zone": zone_num, "south": is_south})
    to_utm = Transformer.from_crs("EPSG:4326", utm_crs, always_xy=True)
    to_wgs = Transformer.from_crs(utm_crs, "EPSG:4326", always_xy=True)

    # Step 5: Project SW corner into UTM
    sw_e, sw_n = to_utm.transform(sw_lon, sw_lat)

    # Step 6: Build the 4 corners (100km tile = 100,000m)
    size = 100_000  # meters
    corners_utm = [
        (sw_e,        sw_n       ),  # SW
        (sw_e + size, sw_n       ),  # SE
        (sw_e + size, sw_n + size),  # NE
        (sw_e,        sw_n + size),  # NW
    ]

    # Step 7: Back to WGS84 (lon, lat)
    corners_wgs84 = [to_wgs.transform(e, n) for e, n in corners_utm]

    return Polygon(corners_wgs84)  # shapely expects (x, y) = (lon, lat)

# ─────────────────────────────────────────────
# MAIN: Process each tile
# ─────────────────────────────────────────────

for tile_id in TILES:
    try:
        polygon = get_mgrs_tile_polygon(tile_id)

        gdf = gpd.GeoDataFrame(
            {"tile": [tile_id]},
            geometry=[polygon],
            crs="EPSG:4326"
        )

        out_path = os.path.join(OUTPUT_DIR, f"{tile_id}.shp")
        gdf.to_file(out_path)
        print(f"✅ Saved: {out_path}  |  Bounds: {polygon.bounds}")

    except Exception as e:
        print(f"⚠️  Error on tile {tile_id}: {e}")

print(f"\nDone! Shapefiles saved to: ./{OUTPUT_DIR}/")

✅ Saved: mgrs_tiles/12TYP.shp  |  Bounds: (-108.53321795123153, 43.293031722989724, -107.24612615875655, 44.22578426349772)
✅ Saved: mgrs_tiles/39KUA.shp  |  Bounds: (49.110189332951165, -18.08639463814509, 50.0596517124647, -17.176048446896466)
✅ Saved: mgrs_tiles/18SUJ.shp  |  Bounds: (-77.33362068382995, 38.82609253750627, -76.15228761458222, 39.74403956295097)
✅ Saved: mgrs_tiles/15TYL.shp  |  Bounds: (-90.45687651021736, 45.08980169318413, -89.12639482385204, 46.02435339582258)
✅ Saved: mgrs_tiles/10SGD.shp  |  Bounds: (-120.82622923842953, 34.297911285926624, -119.70503579673576, 35.223124940555365)
✅ Saved: mgrs_tiles/17PPL.shp  |  Bounds: (-80.09012046806419, 9.042047066405905, -79.17569961634088, 9.949802036095289)

Done! Shapefiles saved to: ./mgrs_tiles/


In [ ]:
# ─────────────────────────────────────────────
# STEP 5: Filter and export each tile
# ─────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

for tile_id in TILES:
    tile_gdf = gdf[gdf[tile_col] == tile_id]

    if tile_gdf.empty:
        print(f"  ⚠️  WARNING: Tile '{tile_id}' not found in dataset. Skipping.")
        continue

    out_path = os.path.join(OUTPUT_DIR, f"{tile_id}.shp")
    tile_gdf.to_file(out_path)
    print(f"  ✅ Saved: {out_path}  (CRS: {tile_gdf.crs})")

print(f"\nDone! Shapefiles saved to: ./{OUTPUT_DIR}/")